# 05 — Recommender Embeddings with Matrix Factorization

This notebook uses unsupervised matrix factorization to learn user and item embeddings from implicit interactions.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd() / "src"))

from unsup_lab.data import make_user_item_interactions
from unsup_lab.plotting import plot_embedding
from unsup_lab.recommenders import (
    build_sparse_interactions,
    discover_item_groups,
    factorize_interactions,
    recommend_for_user,
    similar_items,
)

In [ ]:
interactions, dense_matrix = make_user_item_interactions(
    n_users=600,
    n_items=120,
    n_latent_groups=5,
    random_state=42,
)

# Rebuild the interaction matrix in sparse form from the long-form log,
# which is how it would arrive from an event store in practice.
matrix = build_sparse_interactions(interactions, n_users=600, n_items=120)
print(f"sparse shape: {matrix.shape}, stored entries: {matrix.nnz}")
interactions.head()

## Learn latent factors

In [ ]:
factors = factorize_interactions(matrix, n_components=12, method="svd", random_state=42)
user_factors = factors.user_factors
item_factors = factors.item_factors

print(user_factors.shape)
print(item_factors.shape)

In [ ]:
plot_embedding(user_factors[:, :2], title="User latent space")
plot_embedding(item_factors[:, :2], title="Item latent space")

## Similar item search

In [ ]:
query_item = 3
neighbours = similar_items(item_factors, query_item, k=10)

pd.DataFrame(neighbours, columns=["similar_item", "cosine_similarity"]).assign(
    query_item=query_item
)

## Product group discovery

Clustering the item embeddings groups products that are co-consumed, which is a useful unsupervised proxy for catalogue structure.

In [ ]:
item_groups = discover_item_groups(item_factors, n_groups=5, random_state=42)
pd.Series(item_groups).value_counts().sort_index().rename("n_items").to_frame()

## Recommendations and cold start

Reconstructed scores rank items a user has not yet touched. New users or items with no interactions have no row or column signal, so the model cannot place them - the classic cold-start problem, which needs content features or popularity fallbacks rather than factorisation alone.

In [ ]:
user_id = 0
recommendations = recommend_for_user(factors, matrix, user_id, k=10)
pd.DataFrame(recommendations, columns=["item_id", "score"]).assign(user_id=user_id)

## Limitations

Matrix factorization can reveal useful latent structure, but it can also amplify exposure bias. Implicit feedback is not the same as preference.
